In [1]:
# pip install shared_utils 

In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
from segment_speed_utils.project_vars import PUBLIC_GCS
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.patches import Patch

pd.set_option('display.max_columns', None)

In [3]:
import matplotlib.patches as Patches

In [4]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses'

In [5]:
# Load the stored organization dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/organization_stops_buffered_route_array.parquet", "rb") as f:
    orgs_stop_buffered_route = gpd.read_parquet(f)

In [6]:
# Load the stored organization dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/organization_stops_buffered.parquet", "rb") as f:
    orgs_stop_buffered = gpd.read_parquet(f)

In [7]:
# Load the stored ACS dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/census_tracts_data_2024.parquet", "rb") as f:
    tracts_ca_acs = gpd.read_parquet(f)

In [8]:
# Load Ridership Grouped Data 
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/ridership_data_2024.parquet", "rb") as f:
    ridership_data_grouped = pd.read_parquet(f)

In [9]:
# Load Ridership Grouped Data 
with fs.open(f"{GCS_FILE_PATH}/transit_provider_dashboard/route_id_shapes.parquet", "rb") as f:
    route_id_shapes = gpd.read_parquet(f)

In [10]:
tracts_ca_acs.crs

<Projected CRS: EPSG:3310>
Name: NAD83 / California Albers
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: United States (USA) - California.
- bounds: (-124.45, 32.53, -114.12, 42.01)
Coordinate Operation:
- name: California Albers
- method: Albers Equal Area
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [11]:
orgs_stop_buffered.crs

<Projected CRS: EPSG:3310>
Name: NAD83 / California Albers
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: United States (USA) - California.
- bounds: (-124.45, 32.53, -114.12, 42.01)
Coordinate Operation:
- name: California Albers
- method: Albers Equal Area
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [12]:
orgs_stop_buffered.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 133556 entries, 0 to 133555
Data columns (total 16 columns):
 #   Column                         Non-Null Count   Dtype   
---  ------                         --------------   -----   
 0   name                           133556 non-null  object  
 1   ntd_id_x                       85055 non-null   object  
 2   ntd_id_2022_x                  85312 non-null   object  
 3   stop_id                        133556 non-null  object  
 4   stop_name                      133556 non-null  object  
 5   schedule_gtfs_dataset_name     96325 non-null   object  
 6   organization_source_record_id  97614 non-null   object  
 7   geometry                       133556 non-null  geometry
 8   analysis_name                  97695 non-null   object  
 9   organization_name              96325 non-null   object  
 10  name_clean                     133556 non-null  object  
 11  source_record_id               68069 non-null   object  
 12  key     

In [13]:
route_id_shapes.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 11219 entries, 0 to 11218
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   feed_key  11219 non-null  object  
 1   route_id  11219 non-null  object  
 2   shape_id  11219 non-null  object  
 3   geometry  11219 non-null  geometry
dtypes: geometry(1), object(3)
memory usage: 350.7+ KB


In [14]:
allowed_vctc_intercity = [
    "Thousand Oaks Transit Center", "The Oaks", "Camarillo Metrolink Station", "Carmen Plaza",
    "Camarillo Outlets Food Court", "Esplanade Mall NB", "Ventura Pier",
    "Pacific View Mall (Ventura Transit Center)", "Esplanade Mall SB",
    "Camarillo Outlets - Main Court", "Thousand Oaks Library / Teen Center",
    "Moorpark Station EB", "Princeton Ave at Amherst NB", "Simi Town Center EB",
    "C Street Transfer Center NB", "Oxnard College EB", "CSU Channel Islands",
    "Villa Calleguas", "Oxnard College WB", "Fillmore Active Adult & Community Center",
    "Santa Paula City Hall", "Santa Paula DMV (Harvard & Steckel)",
    "Santa Paula Kmart", "Ventura College (NW corner of Telegraph/Estates Ave)",
    "Ventura College (SW Corner of Telegraph/Estates Ave)",
    "Santa Paula DMV (Harvard at Craig)", "Moorpark Station WB",
    "Santa Clara and Oak WB (Downtown Ventura)", "Carpinteria Ave at Eugenia Pl (Downtown Carp)",
    "Cabrillo at Puerta Vallarta WB (East Beach)", "Gutierrez at Garden",
    "Figueroa at Chapala (MTD Transit Center)", "Bath and Pueblo NB (Cottage Hospital)",
    "SB Airport NB", "UCSB Bus Loop", "SB Airport SB", "Haley at Garden",
    "Cabrillo at Puerta Vallarta EB (East Beach)", "Carpinteria Ave at Maple (Downtown Carp)",
    "Santa Clara and Oak EB (Downtown Ventura)", "Princeton Ave at Amherst SB",
    "Gov Center (NE Telephone/Victoria)", "Buena High School (NW Telegraph/Wake Forest)",
    "VCMC Westbound", "St. Bon HS Westbound", "Saint Bonaventure High School",
    "Ventura County Medical Center", "Buena High School",
    "Gov. Center (SE Telephone/Victoria)", "Ventura County Government Center (Hill Rd/Thille St)",
    "West Main at Peking WB (Park & Ride)", "Camino del Remedio WB (SB County Complex)",
    "Turnpike & San Gordiano (SMHS)", "Hollister and Patterson WB (Goleta College Hospital)",
    "Hollister and Kellogg WB", "Hollister and Nectarine", "Hollister and Pine",
    "Hollister and Kellogg EB", "West Main at Peking EB (Park & Ride)",
    "Ventura County Government Center (NW Victoria Ave/Telephone Rd)",
    "NW Cochran St/Galena St (Simi Valley Park & Ride)", "Simi Town Center WB",
    "Simi Valley Metrolink", "Simi Valley Civic Center - SE",
    "Somis SW corner Somis Rd/Rice St", "Ventura County Government Center (NE Victoria Ave/Telephone Rd)",
    "Somis NE corner Somis Rd/Rice St", "SW Cochran St/Galena St",
    "Simi Valley Civic Center - SW", "Santa Barbara at De La Guerra",
    "Figueroa at Santa Barbara St (Courthouse)", "De La Vina & Mission",
    "Anacapa at Anapamu (SB Library)", "Anacapa at De La Guerra (SB City Hall)",
    "Victory & Topanga", "Victory & De Soto", "Victory & Mason", "Burbank & De Soto",
    "Thousand Oaks Transit Center NB", "Hollister & Palo Alto SB", "Hollister & Entrance SB",
    "Cortona at Castilian", "Castilian at Los Carneros", "Los Carneros Rd/Karl Storz Way NB",
    "Hollister and Aero Camino EB", "Hollister & La Patera SB", "Hollister & La Patera NB",
    "Hollister and Aero Camino WB", "Los Carneros Rd/Karl Storz Way SB",
    "Castilian at Los Carneros NB", "Castilian at Cortona", "Hollister & Entrance NB",
    "Hollister & Palo Alto NB", "Hollister and Patterson EB (Goleta Cottage Hospital)",
    "San Marcos High School"
]

allowed_valley_express_stops = [
    "Fillmore Terminal Inbound", "Fillmore Terminal Outbound", "7-Eleven", "Mercado La Plaza",
    "Faith Community Church", "C St. & Sespe Ave.", "C St. & Meadowlark Drive", "Shiell Park",
    "Mountain Vista School", "4th & B St", "4th & B St (Eastbound)", "Third St. & B St.",
    "City Hall", "Downtown", "Fillmore MS & HS", "Central Ave & 4th St.", "San Cayetano School",
    "Mountain View St. & Third St.", "Los Serenos Drive & Sierra Vista Ave", "Los Serenos Drive",
    "El Paso St. & Sierra Vista Ave.", "Sierra Vista Ave & Sespe Ave", "Sespe Ave. & Burson Ln.",
    "Sespe Ave. & Burson Ln. (Westbound)", "Price St. & First St.", "McCampbell St. & Wileman St.",
    "B St. & Santa Clara St.", "B St. & Santa Clara St. (Northbound)", "River St. & Deerfield Drive",
    "River St. & Deerfield Drive (Eastbound)", "Santa Fe St. & Rio Grande St.", "Surrey Way & Santa Fe St.",
    "Santa Fe St. & Reading St.", "River St. (Westbound)", "Rio Vista Elementary", "Boys & Girls Club",
    "El Dorado Estate", "Rancho Sespe", "Main St. & Shannon Ln (park)", "Piru Square", "Valle Naranjal",
    "Santa Barbara St. & 4th St.", "Santa Barbara St. & 11th St.", "12th St. & Santa Paula St.",
    "Main St. & 12th St.", "Harvard Blvd. & Garcia St.", "Harvard Blvd. & Ojai St.",
    "Harvard Blvd. & Steckel Drive", "Harvard Blvd. & Laurie Ln.", "Harvard Blvd. & Craig Drive",
    "Harvard Blvd. & 8th St.", "10th St. & Santa Barbara St.", "Main St. & 11th St.",
    "Main St. & 4th St.", "Main St. & Mill St.", "10th St. & Virginia Terr.", "Santa Barbara St. & 8th St.",
    "Santa Paula Hospital", "Beckwith Rd. & Via Pasada", "Santa Paula St. & Walden St.",
    "12th St. & Richmond Rd.", "Barbara Webster Elem.", "Bedell School", "McKevett School",
    "Santa Paula High School", "Isbell School", "Glen City School", "Blanchard School",
    "Moorpark College", "Moorpark Metrolink WB", "Moorpark Metrolink EB"
]


allowed_city_of_camarillo_stops = [
    "Leisure Village Club House",
    "East Gate/ Village #44",
    "Santa Rosa Plaza",
    "Plaza at Mission Oaks (Pardee Plaza)",
    "Camarillo Library",
    "Pleasant Valley Hospital",
    "Village Square",
    "Central Plaza",
    "Ponderosa Plaza",
    "Post Office",
    "Community Center",
    "Mira Vista Village",
    "Pleasant Valley Park",
    "Mission Oaks Plaza (Inbound)",
    "Mission Oaks Plaza (Outbound)",
    "East Gate/Village #41",
    "Mtn. View/ Village #23",
    "Mtn. View/Village #24",
    "Mtn. View /Village #25",
    "Mtn. View/ #30",
    "Mtn. View/ Village #4",
    "Mtn. View/Village #16"
]

allowed_city_of_ojai_stops = [
    "Ojai Ave @ Arcade",
    "Ojai Ave & Signal",
    "Ojai Ave @ Westridge Mkt",
    "Rice & Camille",
    "Rice & Fierro",
    "Rice & El Sereno Estates",
    "Rice & Woodland",
    "Rice & Hwy 150 NW Corner",
    "Woodland & Hwy 33",
    "Hwy 33 @ Red Horse Plaza",
    "La Luna @ Fire Station",
    "El Roblar & La Luna",
    "El Roblar & Encinal",
    "El Roblar & Lomita",
    "El Roblar @ Roth Apts",
    "Hospital",
    "Y Memorial Garden",
    "Loma @ Mira Valle MH Park",
    "Ojai Valley Inn",
    "Ojai @ Bank of America",
    "Nordhoff HS"
]


# Identify rows currently assigned to the broad VCTC group.
vctc_parent = (
    "Ventura County (VCTC, Gold Coast, Cities of Camarillo, "
    "Moorpark, Ojai, Simi Valley, Thousand Oaks)"
)
mask_vctc = orgs_stop_buffered["analysis_name"].eq(vctc_parent)

# Map each VCTC subgroup to its allowed stops.
vctc_groups = {
    "VCTC Intercity": allowed_vctc_intercity,
    "City of Camarillo": allowed_city_of_camarillo_stops,
    "City of Ojai": allowed_city_of_ojai_stops,
    "VCTC Valley Express": allowed_valley_express_stops,
}

# Reassign VCTC stops to their specific subgroup.
for group, stops in vctc_groups.items():
    orgs_stop_buffered.loc[
        mask_vctc & orgs_stop_buffered["stop_name"].isin(stops),
        "analysis_name"
    ] = group

# Store NTD and operating statistics by agency.
agency_map_ntd = {
    "City of Ojai": {"ntd_id_2022": "91058", "upt": 48315, "voms": 2},
    "City of Camarillo": {"ntd_id_2022": "90163", "upt": 73504, "voms": 18},
}

# for agency, vals in orgs_stop_buffered.items():
#     mask = orgs_stop_buffered["analysis_name"].eq(agency)

#     orgs_stop_buffered.loc[mask, "ntd_id_2022_y"] = vals["ntd_id_2022_y"]
#     orgs_stop_buffered.loc[mask, "unlinked_passenger_trips_upt"] = vals["upt"]
#     orgs_stop_buffered.loc[mask, "agency_voms"] = vals["voms"]



In [15]:
# Define stops that belong to Metrolink.
allowed_metrolink_stops = [
    "L.A. Union Station", "Cal State LA", "El Monte", "Baldwin Park",
    "Covina", "Pomona - North", "Claremont", "Montclair", "Upland",
    "Rancho Cucamonga", "Fontana", "Rialto", "San Bernardino Depot",
    "San Bernardino - Downtown", "Redlands - University",
    "Redlands - Downtown", "Redlands - Esri", "San Bernardino - Tippecanoe"
]

# Identify rows currently assigned to the Metrolink agency.
metrolink_mask = orgs_stop_buffered["analysis_name"].eq(
    "Southern California Regional Rail Authority"
)

# Keep all other agencies and only allowed stops for Metrolink.
orgs_stop_buffered = orgs_stop_buffered[
    ~metrolink_mask | orgs_stop_buffered["stop_name"].isin(allowed_metrolink_stops)
].copy()

In [16]:
# Reconciliation groups for different organization subset 
RECONCILIATION_GROUPS = {
    "contactless_credit_debit": [
        "Alameda-Contra Costa Transit District",
        "Anaheim Transportation Network",
        "Antelope Valley Transit Authority",
        "Peninsula Corridor Joint Powers Board",
        "Capitol Corridor Joint Powers Authority",
        "City and County of San Francisco",
        "City of Baldwin Park",
        "City of Burbank",
        "City of Carson", 
        "City of Compton",
        "City of Culver City",       
        "City of Fairfield",
        "City of Gardena",
        "City of Glendora",
        "City of Huntington Park",
        "City of Lawndale",
        "City of Montebello",
        "City of Monterey Park",
        "City of Morro Bay",
        "City of Norwalk",
        "City of Pasadena",
        "Redding Area Bus Authority",
        "City of Redondo Beach",
        "City of Santa Clarita",
        "City of Santa Rosa",
        "City of San Luis Obispo",
        "City of Torrance",
        "City of Union City",
        "City of Vacaville",
        "Cloverdale Transit",
        "Central Contra Costa Transit Authority", # County Connection
        "Eastern Contra Costa Transit Authority",
        "El Dorado County Transit Authority",
        "Foothill Transit",  
        "City of Glendale",
        "Golden Gate Bridge",
        "Humboldt Transit Authority", #Humboldt Transit"
        "City of Los Angeles", #LADOT
        "Lake Transit Authority",
        "Livermore-Amador Valley Transit Authority",  #lAVTA
        "Los Angeles County",
        "Los Angeles County Metropolitan Transportation Authority",
        "Los Angeles World Airports",
        "Marin County Transit District",
        "Mendocino Transit Authority",
        "Monterey-Salinas Transit",
        "Napa Valley Transportation Authority",
        "North County Transit District",        
        "Nevada County",
        "Orange County Transportation Authority",
        "Palos Verdes Peninsula Transit Authority",
        "Dumbarton Bridge Regional Operations Consortium",
        "City of Petaluma",        
        "Redwood Coast Transit Authority",
        "Sacramento Regional Transit District",
        "San Diego Metropolitan Transit System, Airport, Flagship Cruises",
        "San Francisco Bay Area Rapid Transit District",
        "San Mateo County Transit District",
        "Santa Clara Valley Transportation Authority",
        "City of Santa Monica",
        "Santa Barbara County Association of Governments", # SBCAG/CAE
        "Santa Barbara Metropolitan Transit District",
        "San Luis Obispo Regional Transit Authority",
        "Solano Transportation Authority",
        "Sonoma County",
        "Sonoma-Marin Area Rail Transit District",
        "VCTC Valley Express",
        "City of Camarillo",
        "VCTC Intercity",
        "Western Contra Costa Transit Authority",
    ],
    
    "contactless_payments_now_ca_msas": [
        "VCTC Valley Express",
        "City of Camarillo",        
        "San Luis Obispo Regional Transit Authority",
        "City of Morro Bay",
        "VCTC Intercity",
        "Redding Area Bus Authority",
        "Redwood Coast Transit Authority",
        "Lake Transit Authority",
        "Humboldt Transit Authority", #Humboldt Transit"
        "Sacramento Regional Transit District",
        "El Dorado County Transit Authority",
        "Capitol Corridor Joint Powers Authority",
        "Santa Barbara Metropolitan Transit District",
        "Santa Barbara County Association of Governments", # SBCAG/CAE
        "Monterey-Salinas Transit",
        "Mendocino Transit Authority",
        "City of San Luis Obispo",
        "Anaheim Transportation Network",
        "Nevada County",
    ],    

    "contactless_q2": [
        "Alameda-Contra Costa Transit District",
        "Anaheim Transportation Network",
        "Antelope Valley Transit Authority",
        "Peninsula Corridor Joint Powers Board",
        "Capitol Corridor Joint Powers Authority",
        "City and County of San Francisco",
        "City of Baldwin Park",
        "City of Burbank",
        "City of Carson", 
        "City of Compton",
        "City of Culver City",       
        "City of Fairfield",
        "City of Gardena",
        "City of Glendora",
        "City of Huntington Park",
        "City of Lawndale",
        "City of Montebello",
        "City of Monterey Park",
        "City of Morro Bay",
        "City of Norwalk",
        "City of Pasadena",
        "Redding Area Bus Authority",
        "City of Redondo Beach",
        "City of Santa Clarita",
        "City of Santa Rosa",
        "City of San Luis Obispo",
        "City of Torrance",
        "City of Union City",
        "City of Vacaville",
        "Cloverdale Transit",
        "Central Contra Costa Transit Authority", # County Connection
        "Eastern Contra Costa Transit Authority",
        "El Dorado County Transit Authority",
        "Foothill Transit",  
        "City of Glendale",
        "Golden Gate Bridge",
        "Humboldt Transit Authority", #Humboldt Transit"
        "City of Los Angeles", #LADOT
        "Lake Transit Authority",
        "Livermore-Amador Valley Transit Authority",  #lAVTA
        "Los Angeles County",
        "Los Angeles County Metropolitan Transportation Authority",
        "Los Angeles World Airports",
        "Marin County Transit District",
        "Mendocino Transit Authority",
        "Monterey-Salinas Transit",
        "Napa Valley Transportation Authority",
        "North County Transit District",        
        "Nevada County",
        "Orange County Transportation Authority",
        "Palos Verdes Peninsula Transit Authority",
        "Dumbarton Bridge Regional Operations Consortium",
        "City of Petaluma",        
        "Redwood Coast Transit Authority",
        "Sacramento Regional Transit District",
        "San Diego Metropolitan Transit System, Airport, Flagship Cruises",
        "San Francisco Bay Area Rapid Transit District",
        "San Mateo County Transit District",
        "Santa Clara Valley Transportation Authority",
        "City of Santa Monica",
        "Santa Barbara County Association of Governments", # SBCAG/CAE
        "Santa Barbara Metropolitan Transit District",
        "San Luis Obispo Regional Transit Authority",
        "Solano Transportation Authority",
        "Sonoma County",
        "Sonoma-Marin Area Rail Transit District",
        "VCTC Valley Express",
        "City of Camarillo",
        "VCTC Intercity",
        "Western Contra Costa Transit Authority",
        "Santa Cruz Metropolitan Transit District",
        "Southern California Regional Rail Authority", # "Metrolink (San Bernadino Line and Arrow Service only)"
    ],



    "contactless_payments_next_six_months": [
        "City of Simi Valley",
        "Gold Coast Transit District",
        "City of Thousand Oaks",
        "City of Moorpark",
        "Yolo County Transportation District",
        "Yuba-Sutter Transit Authority",        
        "City of Ojai",
        "City of Roseville",
        "Glenn County",
        "Stanislaus Regional Transit Authority",
        "South County Transit Link", #Municipal Services Agency, dba: South County Transit",
        "Golden Empire Transit District",
        "Trinity County",
        "Siskiyou County",
        "Butte County Association of Governments",
        #"City of Wasco",
        "Imperial County Transportation Commission",
        "Madera County",
        "SunLine Transit Agency",
    ],

    "contactless_end_of_2026": [
        "Alameda-Contra Costa Transit District",
        "Anaheim Transportation Network",
        "Antelope Valley Transit Authority",
        "Peninsula Corridor Joint Powers Board",
        "Capitol Corridor Joint Powers Authority",
        "City and County of San Francisco",
        "City of Baldwin Park",
        "City of Burbank",
        "City of Carson", 
        "City of Compton",
        "City of Culver City",       
        "City of Fairfield",
        "City of Gardena",
        "City of Glendora",
        "City of Huntington Park",
        "City of Lawndale",
        "City of Montebello",
        "City of Monterey Park",
        "City of Morro Bay",
        "City of Norwalk",
        "City of Pasadena",
        "Redding Area Bus Authority",
        "City of Redondo Beach",
        "City of Santa Clarita",
        "City of Santa Rosa",
        "City of San Luis Obispo",
        "City of Torrance",
        "City of Union City",
        "City of Vacaville",
        "Cloverdale Transit",
        "Central Contra Costa Transit Authority", # County Connection
        "Eastern Contra Costa Transit Authority",
        "El Dorado County Transit Authority",
        "Foothill Transit",  
        "City of Glendale",
        "Golden Gate Bridge",
        "Humboldt Transit Authority", #Humboldt Transit"
        "City of Los Angeles", #LADOT
        "Lake Transit Authority",
        "Livermore-Amador Valley Transit Authority",  #lAVTA
        "Los Angeles County",
        "Los Angeles County Metropolitan Transportation Authority",
        "Los Angeles World Airports",
        "Marin County Transit District",
        "Mendocino Transit Authority",
        "Monterey-Salinas Transit",
        "Napa Valley Transportation Authority",
        "North County Transit District",        
        "Nevada County",
        "Orange County Transportation Authority",
        "Palos Verdes Peninsula Transit Authority",
        "Dumbarton Bridge Regional Operations Consortium",
        "City of Petaluma",        
        "Redwood Coast Transit Authority",
        "Sacramento Regional Transit District",
        "San Diego Metropolitan Transit System, Airport, Flagship Cruises",
        "San Francisco Bay Area Rapid Transit District",
        "San Mateo County Transit District",
        "Santa Clara Valley Transportation Authority",
        "City of Santa Monica",
        "Santa Barbara County Association of Governments", # SBCAG/CAE
        "Santa Barbara Metropolitan Transit District",
        "San Luis Obispo Regional Transit Authority",
        "Solano Transportation Authority",
        "Sonoma County",
        "Sonoma-Marin Area Rail Transit District",
        "VCTC Valley Express",
        "City of Camarillo",
        "VCTC Intercity",
        "Western Contra Costa Transit Authority",
        "Santa Cruz Metropolitan Transit District",
        "Southern California Regional Rail Authority", # "Metrolink (San Bernadino Line and Arrow Service only)"
        "City of Simi Valley",
        "Gold Coast Transit District",
        "City of Thousand Oaks",
        "City of Moorpark",
        "Yolo County Transportation District",
        "Yuba-Sutter Transit Authority",        
        "City of Roseville",
        "Glenn County",
        "Stanislaus Regional Transit Authority",
        "South County Transit Link", #Municipal Services Agency, dba: South County Transit",
        "Golden Empire Transit District",
        "Trinity County",
        "Siskiyou County",
        "Butte County Association of Governments",
        #"City of Wasco",
        "Imperial County Transportation Commission",
        "Madera County",
        "SunLine Transit Agency",    
    ],

    "reduced_fares_live_now": [
        "Monterey-Salinas Transit",
        "Santa Barbara Metropolitan Transit District",
        "Sacramento Regional Transit District",
        "Nevada County",
        "VCTC Intercity",
        "San Luis Obispo Regional Transit Authority",
        "El Dorado County Transit Authority",
        "Redding Area Bus Authority",
        "City of San Luis Obispo",
    ],

    "reduced_fares_next_6_months": [
        "Gold Coast Transit District",
        "Santa Cruz Metropolitan Transit District",  
        "City of Camarillo",
        "City of Roseville",
        "VCTC Valley Express",
        "City of Simi Valley",
        "City of Thousand Oaks",
        "Santa Barbara County Association of Governments",
    ],

    "access_to_reduced_fares_contactless": [
        "Monterey-Salinas Transit",
        "Santa Barbara Metropolitan Transit District",
        "Sacramento Regional Transit District",
        "Nevada County",
        "VCTC Intercity",
        "San Luis Obispo Regional Transit Authority",
        "El Dorado County Transit Authority",
        "Redding Area Bus Authority",
        "City of San Luis Obispo",
        "Gold Coast Transit District",
        "Santa Cruz Metropolitan Transit District", 
        "City of Camarillo",
        "City of Roseville",
        "VCTC Valley Express",
        "City of Simi Valley",
        "City of Thousand Oaks",
        "Santa Barbara County Association of Governments",
    ],  
}

In [17]:
# Get unique agency names currently in the data.
analysis_names = set(orgs_stop_buffered["analysis_name"].dropna().unique())

# Get all agency names listed in the reconciliation groups.
reconciliation_names = {
    name for group in RECONCILIATION_GROUPS.values() for name in group
}

# Find reconciliation names that are missing from the data.
missing = sorted(reconciliation_names - analysis_names)

print(f"{len(missing)} names not found:\n")
for name in missing:
    print(name)

2 names not found:

Sonoma County
Trinity County


In [18]:
output_folder = f"{GCS_FILE_PATH}/transit_provider_dashboard/june_2026/"

In [19]:
cols_to_weight = [
    "total_pop", "poverty_pop", "non_us_citizen",
    "workers_with_no_car", "households_with_no_cars",
    "disabled_pop", "public_asst_pop",
    "inc_extremelylow", "inc_verylow", "inc_low",
    "male_seniors", "female_seniors",
    "male_youth", "female_youth",
    "veteran_pop"
]

In [20]:
GCS__PUBLIC_FILE_PATH = f"{PUBLIC_GCS}transit_provider_dashboard/june_2026/"

### Calculating : Access to any public transit by groups.

In [21]:
route_id_shapes.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [22]:
# # Process each reconciliation group and save ACS/NTD data.
# for group_name, org_list in RECONCILIATION_GROUPS.items():

#     # Get group stops and dissolve into one geometry.
#     subset = orgs_stop_buffered[orgs_stop_buffered["analysis_name"].isin(org_list)]
#     dissolved_geometry = subset.dissolve().reset_index(drop=True)

#     # Intersect group area with Census tracts.
#     intersection = gpd.overlay(dissolved_geometry, tracts_ca_acs, how="intersection", keep_geom_type=True)

#     # Calculate area-weighted ACS values.
#     intersection["area_2"] = intersection.geometry.area
#     intersection["area_ratio"] = intersection["area_2"] / intersection["area_m2"]
#     for col in cols_to_weight:
#         intersection[f"{col}_adj"] = intersection[col] * intersection["area_ratio"]

#     agg_demo = intersection[[f"{c}_adj" for c in cols_to_weight]].sum().to_frame().T

#     # Aggregate NTD data for agencies in the group.
#     ntd_ids = subset["ntd_id_2022_y"].dropna().unique()
#     ntd_subset = ridership_data_grouped[ridership_data_grouped["ntd_id"].isin(ntd_ids)]
#     agg_ntd = ntd_subset[["unlinked_passenger_trips_upt", "agency_voms"]].sum().to_frame().T

#     # Combine ACS, NTD, and geometry.
#     final_gdf = gpd.GeoDataFrame(
#         pd.concat([agg_demo, agg_ntd], axis=1),
#         geometry=[dissolved_geometry.unary_union], crs="EPSG:3310"
#     ).to_crs(4326)

#     # Create output paths.
#     paths = {k: f"{p}{group_name}{ext}" for k, p, ext in [
#         ("parquet", output_folder + "/", ".parquet"),
#         ("geojson", output_folder + "/", ".geojson"),
#         ("csv", output_folder + "/", ".csv"),
#         ("public_parquet", GCS__PUBLIC_FILE_PATH, ".parquet"),
#         ("public_geojson", GCS__PUBLIC_FILE_PATH, ".geojson"),
#         ("public_csv", GCS__PUBLIC_FILE_PATH, ".csv"),
#     ]}

#     # Convert geometry to WKT for CSV.
#     final_gdf_copy = final_gdf.copy()
#     final_gdf_copy["geometry"] = final_gdf_copy.geometry.apply(lambda x: x.wkt if x else None)

#     # Save private and public outputs.
#     for prefix in ["", "public_"]:
#         with fs.open(paths[f"{prefix}parquet"], "wb") as f:
#             final_gdf.to_parquet(f, engine="pyarrow", index=False)
#         with fs.open(paths[f"{prefix}geojson"], "wb") as f:
#             final_gdf.to_file(f, driver="GeoJSON")
#         with fs.open(paths[f"{prefix}csv"], "wb") as f:
#             final_gdf_copy.to_csv(f, index=False)

#     print(f"Uploaded {group_name}")

In [30]:

from matplotlib.patches import Patch


In [ ]:
example_group = "contactless_payments_now_ca_msas"
plot_folder = f"{GCS_FILE_PATH}/transit_provider_dashboard/maps/routes_only"

# California boundary.
ca_counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip")
ca_counties = ca_counties[ca_counties["STATEFP"] == "06"].to_crs(4326)
ca_state = ca_counties.dissolve()
xmin, ymin, xmax, ymax = ca_state.total_bounds

# Census tracts and population density.
tracts_area = tracts_ca_acs.to_crs("EPSG:3310").copy()
tracts_area["area_km2"] = tracts_area.geometry.area / 1_000_000
tracts_area["pop_density"] = tracts_area["total_pop"] / tracts_area["area_km2"]
tracts_plot = tracts_area.to_crs(4326)

# Population density bins.
density_bins = [0, 100, 500, 1000, 2500, 5000, 10000, 20000, 2500000]
density_colors = plt.cm.BuPu(np.linspace(.15, .90, len(density_bins) - 1))
density_cmap = colors.ListedColormap(density_colors)
density_norm = colors.BoundaryNorm(density_bins, density_cmap.N)
density_labels = ["0–100", "100–500", "500–1,000", "1,000–2,500", "2,500–5,000", "5,000–10,000", "10,000–20,000", "20,000+"]
handles = [Patch(facecolor=density_colors[i], edgecolor="none", label=density_labels[i]) for i in range(len(density_labels))]

# Create and save maps for all reconciliation groups.
for group_name, org_list in RECONCILIATION_GROUPS.items():
    subset = orgs_stop_buffered_route[orgs_stop_buffered_route["analysis_name"].isin(org_list)]
    route_ids = subset["route_id_array"].dropna().explode().astype(str).str.strip().unique()
    routes = gpd.clip(route_id_shapes[route_id_shapes["route_id"].astype(str).isin(route_ids)].to_crs(4326), ca_state)

    fig, ax = plt.subplots(figsize=(12, 10))
    tracts_plot.plot(ax=ax, column="pop_density", cmap=density_cmap, norm=density_norm, linewidth=0, alpha=0.7)
    ca_counties.boundary.plot(ax=ax, color="#BDBDBD", linewidth=0.5)

    if not routes.empty:
        routes.plot(ax=ax, color=density_colors[-1], alpha=0.9, linewidth=0.8)

    ax.legend(handles=handles, title="Population density\npeople / km²", loc="lower left",
              fontsize=8, title_fontsize=9, frameon=True, facecolor="white", edgecolor="none")
    ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
    ax.set_title(f"{group_name.replace('_', ' ').title()}\nPopulation Density and Routes Serving Stops", fontsize=16)
    ax.set_axis_off(); plt.tight_layout()

    with fs.open(f"{plot_folder}/{group_name}.png", "wb") as f:
        fig.savefig(f, format="png", dpi=300, bbox_inches="tight")
    if group_name != example_group: plt.close(fig)
    print(f"Saved: {plot_folder}/{group_name}.png")

plt.show()

In [23]:
import numpy as np

In [ ]:
example_group = "contactless_payments_now_ca_msas"
plot_folder = f"{GCS_FILE_PATH}/transit_provider_dashboard/maps/routes_stops"

# California boundary.
ca_counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip")
ca_counties = ca_counties[ca_counties["STATEFP"] == "06"].to_crs(4326)
ca_state = ca_counties.dissolve()
xmin, ymin, xmax, ymax = ca_state.total_bounds

# Census tracts and population density.
tracts_area = tracts_ca_acs.to_crs("EPSG:3310").copy()
tracts_area["area_km2"] = tracts_area.geometry.area / 1_000_000
tracts_area["pop_density"] = tracts_area["total_pop"] / tracts_area["area_km2"]
tracts_plot = tracts_area.to_crs(4326)

# Population density bins.
density_bins = [0, 100, 500, 1000, 2500, 5000, 10000, 20000, 2500000]
density_colors = plt.cm.BuPu(np.linspace(.15, .90, len(density_bins) - 1))
density_cmap = colors.ListedColormap(density_colors)
density_norm = colors.BoundaryNorm(density_bins, density_cmap.N)
density_labels = ["0–100", "100–500", "500–1,000", "1,000–2,500", "2,500–5,000", "5,000–10,000", "10,000–20,000", "20,000+"]
handles = [Patch(facecolor=density_colors[i], edgecolor="none", label=density_labels[i]) for i in range(len(density_labels))]

# Create and save maps for all reconciliation groups.
for group_name, org_list in RECONCILIATION_GROUPS.items():
    subset = orgs_stop_buffered_route[
        orgs_stop_buffered_route["analysis_name"].isin(org_list)
    ].to_crs(4326)

    route_ids = subset["route_id_array"].dropna().explode().astype(str).str.strip().unique()
    routes = gpd.clip(
        route_id_shapes[route_id_shapes["route_id"].astype(str).isin(route_ids)].to_crs(4326),
        ca_state
    )

    print(group_name, "stops:", len(subset), "CRS:", subset.crs)

    fig, ax = plt.subplots(figsize=(12, 10))
    tracts_plot.plot(ax=ax, column="pop_density", cmap=density_cmap, norm=density_norm, linewidth=0, alpha=.7)
    ca_counties.boundary.plot(ax=ax, color="#BDBDBD", linewidth=.5)

    if not subset.empty:
        subset.plot(ax=ax, color="#D9A400", alpha=.45, edgecolor="#D9A400", linewidth=.2)

    if not routes.empty:
        routes.plot(ax=ax, color=density_colors[-1], alpha=.9, linewidth=1.2)

    ax.legend(handles=handles, title="Population density\npeople / km²", loc="lower left",
              fontsize=8, title_fontsize=9, frameon=True, facecolor="white", edgecolor="none")
    ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
    ax.set_title(f"{group_name.replace('_', ' ').title()}\nPopulation Density, 5-Mile Stop Buffers, and Routes", fontsize=16)
    ax.set_axis_off(); plt.tight_layout()

    with fs.open(f"{plot_folder}/{group_name}.png", "wb") as f:
        fig.savefig(f, format="png", dpi=300, bbox_inches="tight")

    if group_name != example_group:
        plt.close(fig)

    print(f"Saved: {plot_folder}/{group_name}.png")

In [ ]:
import numpy as np
example_group = "contactless_payments_now_ca_msas"

plot_folder = f"{output_folder}/maps"
density_folder, route_folder = f"{GCS_FILE_PATH}{plot_folder}/density_only", f"{GCS_FILE_PATH}{plot_folder}/density_routes"
os.makedirs(density_folder, exist_ok=True)
os.makedirs(route_folder, exist_ok=True)

# California boundary
ca_counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip")
ca_counties = ca_counties[ca_counties["STATEFP"] == "06"].to_crs(4326)
ca_state = ca_counties.dissolve()

xmin, ymin, xmax, ymax = ca_state.total_bounds

# Census tracts and population density
tracts_area = tracts_ca_acs.to_crs("EPSG:3310").copy()
tracts_area["area_km2"] = tracts_area.geometry.area / 1_000_000
tracts_area["pop_density"] = tracts_area["total_pop"] / tracts_area["area_km2"]
tracts_plot = tracts_area.to_crs(4326)

# Aesthetic palette
BASE_FILL = "#FAF8F4"
COUNTY_LINE = "#B8C0CC"
STATE_EDGE = "#333333"

# Population density bins
density_bins = [0, 100, 500, 1000, 2500, 5000, 10000, 20000, np.inf]
density_colors = plt.cm.BuPu(np.linspace(.15, .90, len(density_bins) - 1))
density_cmap = colors.ListedColormap(density_colors)
density_norm = colors.BoundaryNorm(density_bins, density_cmap.N)

# darkest purple from density colormap
ROUTE_COLOR = density_colors[-1]

density_labels = [
    "0–100", "100–500", "500–1,000", "1,000–2,500",
    "2,500–5,000", "5,000–10,000", "10,000–20,000", "20,000+"
]

handles = [Patch(facecolor=density_colors[i], edgecolor="none", label=density_labels[i]) for i in range(len(density_labels))]

# LOOP THROUGH RECONCILIATION GROUPS
for group_name, org_list in RECONCILIATION_GROUPS.items():

    subset = orgs_stop_buffered[orgs_stop_buffered["analysis_name"].isin(org_list)]
    route_ids = subset["route_id_array"].dropna().explode().astype(str).str.strip().unique()

    routes = gpd.clip(
        route_id_shapes[route_id_shapes["route_id"].astype(str).isin(route_ids)].to_crs(4326),
        ca_state
    )

    # BEFORE — density only (NO BUFFERED STOPS)
    fig1, ax1 = plt.subplots(figsize=(8, 6))

    ca_state.plot(ax=ax1, color=BASE_FILL, edgecolor=STATE_EDGE, linewidth=1)
    tracts_plot.plot(ax=ax1, column="pop_density", cmap=density_cmap, norm=density_norm, linewidth=0, alpha=.80)
    ca_counties.boundary.plot(ax=ax1, color=COUNTY_LINE, linewidth=.5)

    ax1.legend(handles=handles, title="Population density\npeople / km²", loc="lower left", 
               fontsize=8, title_fontsize=9, frameon=True, facecolor="white", edgecolor="none")

    ax1.set_xlim(xmin, xmax)
    ax1.set_ylim(ymin, ymax)
    ax1.set_axis_off()

    ax1.set_title(f"Before — {group_name.replace('_', ' ').title()}", fontsize=14, color="#222222")

    plt.tight_layout()
    fig1.savefig(f"{density_folder}/{group_name}.png", dpi=300, bbox_inches="tight")
    if group_name == example_group:
        display(fig1)
    plt.close(fig1)

    # ------------------------------------------------------
    # AFTER — density + routes + buffered stops
    # ------------------------------------------------------
    fig2, ax2 = plt.subplots(figsize=(8, 6))

    ca_state.plot(ax=ax2, color=BASE_FILL, edgecolor=STATE_EDGE, linewidth=1)
    tracts_plot.plot(ax=ax2, column="pop_density", cmap=density_cmap, norm=density_norm, linewidth=0, alpha=.80)
    ca_counties.boundary.plot(ax=ax2, color=COUNTY_LINE, linewidth=.5)

    # ★ Buffered stops ONLY in AFTER
    if not subset.empty:
        subset.plot(ax=ax2, color="#D9A400", alpha=0.30, linewidth=0)

    # ★ Routes using darkest purple from density colormap
    if not routes.empty:
        routes.plot(ax=ax2, color=ROUTE_COLOR, alpha=.95, linewidth=1.0)

    ax2.set_xlim(xmin, xmax)
    ax2.set_ylim(ymin, ymax)
    ax2.set_axis_off()

    ax2.set_title(
        f"After — {group_name.replace('_', ' ').title()}", fontsize=14, color="#222222")

    plt.tight_layout()
    fig2.savefig(f"{route_folder}/{group_name}.png", dpi=300, bbox_inches="tight")
    if group_name == example_group:
        display(fig2)
    plt.close(fig2)

    print(f"Saved: {group_name}")

In [ ]:
# from matplotlib.animation import FuncAnimation
# from IPython.display import HTML, display

# group_name = example_group
# org_list = RECONCILIATION_GROUPS[group_name]
# subset = orgs_stop_buffered[orgs_stop_buffered["analysis_name"].isin(org_list)].to_crs(4326)
# route_ids = subset["route_id_array"].dropna().explode().astype(str).str.strip().unique()
# routes = gpd.clip(route_id_shapes[route_id_shapes["route_id"].astype(str).isin(route_ids)].to_crs(4326), ca_state)

# fig, ax = plt.subplots(figsize=(9, 6))

# def draw_base():
#     ca_state.plot(ax=ax, color=BASE_FILL, edgecolor=STATE_EDGE, linewidth=1)
#     tracts_plot.plot(ax=ax, column="pop_density", cmap=density_cmap, norm=density_norm, linewidth=0, alpha=.80)
#     ca_counties.boundary.plot(ax=ax, color=COUNTY_LINE, linewidth=.5)
#     ax.legend(handles=handles, title="Population density\npeople / km²",
#               loc="lower left", bbox_to_anchor=(1.02, 0),
#               fontsize=8, title_fontsize=9, frameon=True,
#               facecolor="white", edgecolor="none")
#     ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax); ax.set_axis_off()

# def update(frame):
#     ax.clear()
#     draw_base()

#     alpha = max(0, min(1, (frame - 30) / 30))
#     title = "Before" if alpha == 0 else "After"

#     if alpha > 0:
#         if not subset.empty:
#             subset.plot(ax=ax, color="#D9A400", alpha=.30 * alpha, linewidth=0)
#         if not routes.empty:
#             routes.plot(ax=ax, color=ROUTE_COLOR, alpha=.95 * alpha, linewidth=1.0)

#     ax.set_title(f"{title} — {group_name.replace('_', ' ').title()}", fontsize=14, color="#222222")
#     return []

# update(0)
# fig.canvas.draw()

# anim = FuncAnimation(fig, update, frames=75, interval=60, repeat=True)
# animation_path = f"{plot_folder}/{group_name}_before_after.html"

# html = anim.to_jshtml()
# with open(animation_path, "w") as f:
#     f.write(html)

# display(HTML(html))
# plt.close(fig)

# print(f"Saved: {animation_path}")

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

BASE_FILL = "#FAF8F4"
COUNTY_LINE = "#B8C0CC"
STATE_EDGE = "#333333"
ROUTE_COLOR = density_colors[-1]

animation_folder = f"{GCS_FILE_PATH}/transit_provider_dashboard/maps/animation"

for group_name, org_list in RECONCILIATION_GROUPS.items():
    subset = orgs_stop_buffered[orgs_stop_buffered["analysis_name"].isin(org_list)].to_crs(4326)

    fig, ax = plt.subplots(figsize=(9, 6))

    def draw_base():
        ca_state.plot(ax=ax, color=BASE_FILL, edgecolor=STATE_EDGE, linewidth=1)
        tracts_plot.plot(ax=ax, column="pop_density", cmap=density_cmap, norm=density_norm,
                         linewidth=0, alpha=.80)
        ca_counties.boundary.plot(ax=ax, color=COUNTY_LINE, linewidth=.5)
        ax.legend(handles=handles, title="Population density\npeople / km²",
                  loc="lower left", bbox_to_anchor=(1.02, 0), fontsize=8,
                  title_fontsize=9, frameon=True, facecolor="white", edgecolor="none")
        ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax); ax.set_axis_off()

    def update(frame):
        ax.clear()
        draw_base()
        alpha = max(0, min(1, (frame - 30) / 30))
        title = "Before" if alpha == 0 else "After"
        if alpha > 0 and not subset.empty:
            subset.plot(ax=ax, color=ROUTE_COLOR, alpha=.95 * alpha,
                        edgecolor=ROUTE_COLOR, linewidth=.2)
        ax.set_title(
            f"{title} — {group_name.replace('_', ' ').title()}\n"
            "Population Density and 5-Mile Stop Buffers",
            fontsize=14, color="#222222"
        )
        return []

    update(0)
    fig.canvas.draw()

    anim = FuncAnimation(fig, update, frames=75, interval=60, repeat=True)
    html = anim.to_jshtml()
    animation_path = f"{animation_folder}/{group_name}_before_after.html"

    with fs.open(animation_path, "w") as f:
        f.write(html)

    if group_name == example_group:
        display(HTML(html))

    plt.close(fig)
    print(f"Saved: {animation_path}")